In [1]:
from pathlib import Path

import pandas as pd

preferred_paths = [
    # "results/20260224_081845_tpch_strategy_usage.csv",
    # "results/20260224_083230_ceb_strategy_usage.csv",
    # "../classify_bespoke_execution/results/20260224_080100_ceb_execution_strategy_usage.csv",
    # "../classify_bespoke_execution/results/20260224_080513_tpch_execution_strategy_usage.csv",
    "../classify_bespoke_execution/results/20260601_220932_tpch_execution_strategy_usage.csv",
    "../classify_bespoke_execution/results/20260601_221550_ceb_execution_strategy_usage.csv",
    "../classify_bespoke_multi_threading/results/20260601_222020_ceb_multi_threading_strategy_usage.csv",
    "../classify_bespoke_multi_threading/results/20260601_222231_tpch_multi_threading_strategy_usage.csv",
    "results/20260601_222916_tpch_strategy_usage.csv",
    "results/20260601_223714_ceb_strategy_usage.csv",
]

search_roots = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]


def resolve_input_path(relative_path: str) -> Path | None:
    rel = Path(relative_path)
    candidate_names = [rel.name, rel.name.replace("strategy", "stragegy")]
    for root in search_roots:
        base_dir = (root / rel.parent).resolve()
        for name in candidate_names:
            candidate = base_dir / name
            if candidate.exists():
                return candidate
    return None


resolved_paths = []
for p in preferred_paths:
    rp = resolve_input_path(p)
    if rp is None:
        raise FileNotFoundError(f"Missing expected file: {p}")
    resolved_paths.append(rp)

dfs = []
for path in resolved_paths:
    df = pd.read_csv(path)
    if "classify_bespoke_execution" in path.parts and "count_unit" not in df.columns:
        df["count_unit"] = "query"

    if "total" in df.columns and "total_queries" in df.columns:
        df["total"] = df["total"].fillna(df["total_queries"])
        df = df.drop(columns=["total_queries"])
    elif "total_queries" in df.columns and "total" not in df.columns:
        df = df.rename(columns={"total_queries": "total"})

    df["source_file"] = path.name
    dfs.append(df)

fused_df = pd.concat(dfs, ignore_index=True)

print(f"Loaded {len(dfs)} files")
print("Sources:", [p.name for p in resolved_paths])
fused_df.head()

Loaded 6 files
Sources: ['20260601_220932_tpch_execution_strategy_usage.csv', '20260601_221550_ceb_execution_strategy_usage.csv', '20260601_222020_ceb_multi_threading_strategy_usage.csv', '20260601_222231_tpch_multi_threading_strategy_usage.csv', '20260601_222916_tpch_strategy_usage.csv', '20260601_223714_ceb_strategy_usage.csv']


,group,strategy,count,fraction,total,count_unit,source_file
0,Join Operators,bitmap_semi_join,6,0.272727,22,query,20260601_220932_tpch_execution_strategy_usage.csv
1,Join Operators,index_nested_loop_join,12,0.545455,22,query,20260601_220932_tpch_execution_strategy_usage.csv
2,Join Operators,tag_array_join,7,0.318182,22,query,20260601_220932_tpch_execution_strategy_usage.csv
3,Join Operators,hash_join,1,0.045455,22,query,20260601_220932_tpch_execution_strategy_usage.csv
4,Join Operators,sort_merge_join,0,0.000000,22,query,20260601_220932_tpch_execution_strategy_usage.csv


In [ ]:
# Merge statistics across files by group/strategy
import sys
from pathlib import Path

repo_root = Path.cwd().parent.parent.parent
if repo_root and str(repo_root) not in sys.path:
    sys.path.append(str(repo_root))
from demo_and_analysis.plots.classify_bespoke_execution.strategy_display_names import (
    STRATEGY_DISPLAY_NAMES,
)

key_cols = ["group", "strategy"]
if "count_unit" in fused_df.columns:
    key_cols.append("count_unit")

numeric_cols = fused_df.select_dtypes(include=["number"]).columns.tolist()
sum_cols = [c for c in numeric_cols if c != "fraction"]

merged_stats = fused_df.groupby(key_cols, as_index=False, dropna=False)[sum_cols].sum()

if {"count", "total"}.issubset(merged_stats.columns):
    merged_stats["fraction"] = merged_stats["count"] / merged_stats["total"]

sort_cols = [c for c in ["group", "count", "strategy"] if c in merged_stats.columns]
ascending = [True, False, True][: len(sort_cols)]
if sort_cols:
    merged_stats = merged_stats.sort_values(sort_cols, ascending=ascending).reset_index(
        drop=True
    )

if "strategy" in merged_stats.columns:
    merged_stats["strategy_description"] = (
        merged_stats["strategy"]
        .map(lambda x: STRATEGY_DISPLAY_NAMES[x][0])
        .fillna(merged_stats["strategy"])
    )
    merged_stats["strategy_ce"] = (
        merged_stats["strategy"]
        .map(lambda x: STRATEGY_DISPLAY_NAMES[x][1])
        .fillna(merged_stats["strategy"])
    )


preferred_col_order = [
    "group",
    "strategy",
    "strategy_description",
    "count_unit",
    "count",
    "total",
    "fraction",
]
ordered_cols = [c for c in preferred_col_order if c in merged_stats.columns]
remaining_cols = [c for c in merged_stats.columns if c not in ordered_cols]
merged_stats = merged_stats[ordered_cols + remaining_cols]

# Dark-mode friendly highlighting (set to False if you prefer light theme colors)
dark_mode = False
if dark_mode:
    palette = ["#1f2937", "#0f3d2e", "#3f2f1d", "#312e81", "#3f1d2e", "#164e63"]
    text_color = "#f3f4f6"
    border_color = "#4b5563"
else:
    palette = ["#f0f9ff", "#f0fdf4", "#fff7ed", "#faf5ff", "#fef2f2", "#ecfeff"]
    text_color = "#111827"
    border_color = "#d1d5db"

unique_groups = (
    merged_stats["group"].drop_duplicates().tolist()
    if "group" in merged_stats.columns
    else []
)
group_to_color = {g: palette[i % len(palette)] for i, g in enumerate(unique_groups)}


def color_by_group(row):
    color = group_to_color.get(row.get("group"), "")
    style = (
        f"background-color: {color}; color: {text_color}; border-color: {border_color}"
        if color
        else ""
    )
    return [style] * len(row)


print(f"Merged rows: {len(merged_stats):,}")
if "group" in merged_stats.columns:
    styled = merged_stats.style.apply(color_by_group, axis=1).set_table_styles(
        [
            {
                "selector": "th",
                "props": [("color", text_color), ("border-color", border_color)],
            },
            {"selector": "td", "props": [("border-color", border_color)]},
        ]
    )
    display(styled)
else:
    merged_stats

Merged rows: 84


,group,strategy,strategy_description,count_unit,count,total,fraction,strategy_ce
0,Aggregation & Ordering Strategies,inline_aggregation,Pipeline-Fused Aggregation,query,32,38,0.842105,E
1,Aggregation & Ordering Strategies,scalar_aggregation,Scalar Aggregation,query,14,38,0.368421,C
2,Aggregation & Ordering Strategies,dense_key_aggregation,Bounded-Domain Hash Aggregation,query,10,38,0.263158,E
3,Aggregation & Ordering Strategies,hash_aggregation,Hash Aggregation,query,9,38,0.236842,C
4,Aggregation & Ordering Strategies,direct_array_aggregation,Array-Indexed Group Aggregation,query,5,38,0.131579,E
5,Aggregation & Ordering Strategies,scalar_std_sort,Single-Threaded std::sort,query,4,38,0.105263,C
6,Aggregation & Ordering Strategies,two_phase_aggregation,Two-Phase Aggregation,query,4,38,0.105263,C
7,Aggregation & Ordering Strategies,pairwise_merge_tree_sort,Per-Thread Sort + Pairwise Merge Tree,query,2,38,0.052632,E
8,Aggregation & Ordering Strategies,parallel_radix_sort,Parallel MSD Radix Sort,query,0,38,0.000000,E
9,Column Encoding Strategies,SoA_columnar_layout,Columnar Layout (Struct-of-Arrays),columns,125,233,0.536481,C


In [ ]:
# filter merged_stats
filtered = merged_stats[merged_stats["count"] > 0]

# group ordering
group_ordering = [
    "Column Encoding Strategies",
    "Query-Support Structures",
    "Scan & Filter Strategies",
    "Join Operators",
    "Aggregation Strategies",
    "Low-Level Execution Optimizations",
    "Coordination & Dispatch",
    "Loader-Side Parallelism",
    "Memory & Cache Discipline",
    "Work Distribution",
]

# Rebuild group_to_color based on display order so consecutive groups never share a color
group_to_color = {g: palette[i % len(palette)] for i, g in enumerate(group_ordering)}

# sort by group-ordering and then count
filtered["group"] = pd.Categorical(
    filtered["group"], categories=group_ordering, ordered=True
)
filtered = filtered.sort_values(
    ["group", "count"], ascending=[True, False]
).reset_index(drop=True)

# skip list
skip_strategies = [
    "full_table_scan",
    "SoA_columnar_layout",
    "temporal_sharding_zonemap",
]
filtered = filtered[~filtered["strategy"].isin(skip_strategies)]

In [12]:
filtered

,group,strategy,strategy_description,count_unit,count,total,fraction,strategy_ce
1,Column Encoding Strategies,sorted_physical_layout,Physical Row Sort Order,columns,45,233,0.193133,C
2,Column Encoding Strategies,dictionary_encoding,Dictionary Encoding,columns,23,233,0.098712,C
3,Column Encoding Strategies,scaled_integer,Scaled Integer (Fixed-Point Decimal),columns,22,233,0.094421,C
4,Column Encoding Strategies,compact_int16_date,Frame-of-Reference Date Encoding,columns,6,233,0.025751,C
5,Column Encoding Strategies,bit_packed_record,Bit-Packed Record (sub-byte fields),columns,5,233,0.021459,E
...,...,...,...,...,...,...,...,...
70,NaN,hash_aggregation,Hash Aggregation,query,9,38,0.236842,C
71,NaN,direct_array_aggregation,Array-Indexed Group Aggregation,query,5,38,0.131579,E
72,NaN,scalar_std_sort,Single-Threaded std::sort,query,4,38,0.105263,C
73,NaN,two_phase_aggregation,Two-Phase Aggregation,query,4,38,0.105263,C


In [15]:
import re

# Build LaTeX table with grouped coloring and rotated group labels on the left.
# Requires in your LaTeX preamble:
# \usepackage[table]{xcolor}
# \usepackage{multirow}
# \usepackage{graphicx}

group_line_breaks = {
    "Aggregation Strategies": ["Aggregation", "Strategies"],
    "Column Encoding Strategies": ["Column", "Encoding", "Strategies"],
    "Join Operators": ["Join", "Operators"],
    "Low-Level Execution Optimizations": ["Low-Level", "Execution", "Optimizations"],
    "Query-Support Structures": ["Query-", "Support", "Structures"],
    "Scan & Filter Strategies": ["Scan", "& Filter", "Strategies"],
    "Coordination & Dispatch": ["Coordination", "& Dispatch"],
    "Loader-Side Parallelism": ["Loader-Side", "Parallelism"],
    "Memory & Cache Discipline": ["Memory", "& Cache", "Discipline"],
    "Work Distribution": ["Work", "Distribution"],
}

strategy_line_breaks = {
    "Columnar Layout (Struct-of-Arrays)": ["Columnar Layout (simple array)"],
    "Precomputed Arithmetic Lookup Table": ["Precomp. Arith. Lookup Table"],
}

count_unit_display = {
    "queries": "/queries",
    "query": "/queries",
    "columns": "/cols",
    "build": "/builds",
}

categorization = {}


latex_df = filtered.copy()

if "strategy_description" not in latex_df.columns and "strategy" in latex_df.columns:
    latex_df["strategy_description"] = (
        latex_df["strategy"]
        .map(lambda x: STRATEGY_DISPLAY_NAMES[x][0])
        .fillna(latex_df["strategy"])
    )
    latex_df["strategy_ce"] = (
        latex_df["strategy"]
        .map(lambda x: STRATEGY_DISPLAY_NAMES[x][1])
        .fillna(latex_df["strategy"])
    )

if "count_unit" not in latex_df.columns:
    latex_df["count_unit"] = ""


def fmt_fraction_with_unit(row):
    frac = row.get("fraction")
    if pd.isna(frac):
        frac_txt = ""
    else:
        frac_txt = f"{frac:.2%}"
    unit_raw = str(row.get("count_unit", "")).strip()
    unit = count_unit_display[unit_raw]
    return f"{frac_txt}{unit}".strip()


latex_df["fraction_with_unit"] = latex_df.apply(fmt_fraction_with_unit, axis=1)
latex_df = latex_df[
    ["group", "strategy_description", "strategy_ce", "fraction_with_unit"]
].copy()


def latex_escape(value: object) -> str:
    s = "" if pd.isna(value) else str(value)
    s = s.replace("\\", r"\textbackslash{}")
    subs = {
        "&": r"\&",
        "%": r"\%",
        "$": r"\$",
        "#": r"\#",
        "_": r"\_",
        "{": r"\{",
        "}": r"\}",
        "~": r"\textasciitilde{}",
        "^": r"\textasciicircum{}",
    }
    return re.sub(r"[&%$#_{}~^]", lambda m: subs[m.group(0)], s)


def hex_to_rgb01(hex_color: str):
    h = hex_color.strip().lstrip("#")
    if len(h) != 6:
        return (1.0, 1.0, 1.0)
    return tuple(int(h[i : i + 2], 16) / 255.0 for i in (0, 2, 4))


groups = latex_df["group"].drop_duplicates().tolist()
latex_color_defs = []
for grp_name in groups:
    color_name = f"grp{abs(hash(grp_name)) % 10**8}"
    r, gch, b = hex_to_rgb01(group_to_color.get(grp_name, "#ffffff"))
    latex_color_defs.append((grp_name, color_name, r, gch, b))

group_to_latex_color = {grp_name: c for grp_name, c, *_ in latex_color_defs}
row_end = "\\\\"
lines = []
lines.append(
    r"% ----- Copy into your LaTeX document (move the definecolor outside the \begin{table}-----"
)
for _, cname, r, gch, b in latex_color_defs:
    lines.append(f"\\definecolor{{{cname}}}{{rgb}}{{{r:.3f},{gch:.3f},{b:.3f}}}")
lines.append("\\begin{tabular}{l l c r}")
lines.append("\\hline")
lines.append(
    f" & \\shortstack[c]{{Bespoke and Non-Bespoke\\\\Database Strategies}} & \\shortstack[c]{{Classical\\\\/Exotic}} & used in \\% of {row_end}"
)
lines.append("\\hline")

colorize: bool = True

for grp, grp_df in latex_df.groupby("group", sort=False, dropna=False):
    rows = len(grp_df)
    color_name = group_to_latex_color.get(grp, "white")
    raw_grp = "" if pd.isna(grp) else str(grp)
    group_label_parts = group_line_breaks.get(raw_grp)
    if group_label_parts:
        escaped_parts = [latex_escape(part) for part in group_label_parts]
        group_label = "\\shortstack{" + r"\\".join(escaped_parts) + "}"
    else:
        group_label = latex_escape(raw_grp)

    for i, (_, row_data) in enumerate(grp_df.iterrows()):
        raw_strat = (
            ""
            if pd.isna(row_data["strategy_description"])
            else str(row_data["strategy_description"])
        )
        strat_label_parts = strategy_line_breaks.get(raw_strat)
        if strat_label_parts:
            escaped_strat_parts = [latex_escape(part) for part in strat_label_parts]
            strat = "\\shortstack[l]{" + r"\\".join(escaped_strat_parts) + "}"
        else:
            strat = latex_escape(raw_strat)
        frac_unit = latex_escape(row_data["fraction_with_unit"])

        if colorize:
            cell_color_str = f"\\cellcolor{{{color_name}}}"
        else:
            cell_color_str = ""

        if rows == 1:
            left = f"{cell_color_str}\\rotatebox[origin=c]{{90}}{{{group_label}}}"
        elif i == rows - 1:
            left = f"{cell_color_str}\\multirow{{-{rows}}}{{*}}{{\\rotatebox[origin=c]{{90}}{{{group_label}}}}}"
        else:
            left = f"{cell_color_str} "
        lines.append(
            f"{left} & {cell_color_str} {strat} & {cell_color_str} {row_data['strategy_ce']} & {cell_color_str} {frac_unit} {row_end}"
        )
    lines.append("\\hline")

lines.append("\\end{tabular}")
latex_table = "\n".join(lines)

print(latex_table)
latex_table

% ----- Copy into your LaTeX document (move the definecolor outside the \begin{table}-----
\definecolor{grp84084837}{rgb}{0.941,0.992,0.957}
\definecolor{grp17438910}{rgb}{0.941,0.992,0.957}
\definecolor{grp98533077}{rgb}{1.000,0.969,0.929}
\definecolor{grp31451547}{rgb}{0.980,0.961,1.000}
\definecolor{grp98550904}{rgb}{0.925,0.996,1.000}
\definecolor{grp52591799}{rgb}{1.000,0.969,0.929}
\definecolor{grp59101408}{rgb}{0.996,0.949,0.949}
\definecolor{grp25051932}{rgb}{0.941,0.976,1.000}
\definecolor{grp91725394}{rgb}{0.980,0.961,1.000}
\definecolor{grp76674645}{rgb}{1.000,1.000,1.000}
\begin{tabular}{l l c r}
\hline
 & \shortstack[c]{Bespoke and Non-Bespoke\\Database Strategies} & \shortstack[c]{Classical\\/Exotic} & used in \% of \\
\hline
\cellcolor{grp84084837}  & \cellcolor{grp84084837} Physical Row Sort Order & \cellcolor{grp84084837} C & \cellcolor{grp84084837} 19.31\%/cols \\
\cellcolor{grp84084837}  & \cellcolor{grp84084837} Dictionary Encoding & \cellcolor{grp84084837} C & \cel

'% ----- Copy into your LaTeX document (move the definecolor outside the \\begin{table}-----\n\\definecolor{grp84084837}{rgb}{0.941,0.992,0.957}\n\\definecolor{grp17438910}{rgb}{0.941,0.992,0.957}\n\\definecolor{grp98533077}{rgb}{1.000,0.969,0.929}\n\\definecolor{grp31451547}{rgb}{0.980,0.961,1.000}\n\\definecolor{grp98550904}{rgb}{0.925,0.996,1.000}\n\\definecolor{grp52591799}{rgb}{1.000,0.969,0.929}\n\\definecolor{grp59101408}{rgb}{0.996,0.949,0.949}\n\\definecolor{grp25051932}{rgb}{0.941,0.976,1.000}\n\\definecolor{grp91725394}{rgb}{0.980,0.961,1.000}\n\\definecolor{grp76674645}{rgb}{1.000,1.000,1.000}\n\\begin{tabular}{l l c r}\n\\hline\n & \\shortstack[c]{Bespoke and Non-Bespoke\\\\Database Strategies} & \\shortstack[c]{Classical\\\\/Exotic} & used in \\% of \\\\\n\\hline\n\\cellcolor{grp84084837}  & \\cellcolor{grp84084837} Physical Row Sort Order & \\cellcolor{grp84084837} C & \\cellcolor{grp84084837} 19.31\\%/cols \\\\\n\\cellcolor{grp84084837}  & \\cellcolor{grp84084837} Dicti

In [14]:
latex_df

,group,strategy_description,strategy_ce,fraction_with_unit
1,Column Encoding Strategies,Physical Row Sort Order,C,19.31%/cols
2,Column Encoding Strategies,Dictionary Encoding,C,9.87%/cols
3,Column Encoding Strategies,Scaled Integer (Fixed-Point Decimal),C,9.44%/cols
4,Column Encoding Strategies,Frame-of-Reference Date Encoding,C,2.58%/cols
5,Column Encoding Strategies,Bit-Packed Record (sub-byte fields),E,2.15%/cols
...,...,...,...,...
70,NaN,Hash Aggregation,C,23.68%/queries
71,NaN,Array-Indexed Group Aggregation,E,13.16%/queries
72,NaN,Single-Threaded std::sort,C,10.53%/queries
73,NaN,Two-Phase Aggregation,C,10.53%/queries
